In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# ==========================================
# 1. LOAD DATA & INITIALIZE SQLITE DATABASE
# ==========================================
excel_file = "DataSet.xlsx"

# Read sheets into DataFrames
sales_df = pd.read_excel(excel_file, sheet_name="SalesItems")
product_df = pd.read_excel(excel_file, sheet_name="ProductItems")

# Create SQLite Database in memory
conn = sqlite3.connect(":memory:")

# Write DataFrames to SQL tables
sales_df.to_sql("SalesItems", conn, if_exists="replace", index=False)
product_df.to_sql("ProductItems", conn, if_exists="replace", index=False)


# Helper function to run SQL queries and print formatted results
def run_query(title, query):
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)
    df_result = pd.read_sql_query(query, conn)
    display(df_result)
    print("\n")
    return df_result


# ==========================================
# 2. RUN SQL QUERIES
# ==========================================

# Query 1: Overall KPIs
q1_kpi = """
SELECT 
    COUNT(DISTINCT TransactionID) AS total_orders,
    COUNT(DISTINCT CustomerID) AS total_customers,
    SUM(TotalAmount) AS total_revenue,
    ROUND(AVG(TotalAmount), 2) AS avg_order_value
FROM SalesItems;
"""
df_kpi = run_query("1. Overall Business KPIs", q1_kpi)

# Query 2: Performance by Category
q2_category = """
SELECT 
    p.Category,
    COUNT(s.TransactionID) AS total_units_sold,
    SUM(s.TotalAmount) AS revenue,
    ROUND(AVG(s.Discount), 2) AS avg_discount_rate
FROM SalesItems s
JOIN ProductItems p ON s.ProductID = p.ProductID
GROUP BY p.Category
ORDER BY revenue DESC;
"""
df_category = run_query("2. Sales Performance by Product Category", q2_category)

# Query 3: Monthly Sales Trend
q3_monthly = """
SELECT 
    strftime('%Y-%m', s.TransactionDate) AS sales_month,
    COUNT(DISTINCT s.TransactionID) AS monthly_orders,
    SUM(s.TotalAmount) AS monthly_revenue
FROM SalesItems s
GROUP BY sales_month
ORDER BY sales_month ASC;
"""
df_monthly = run_query("3. Monthly Revenue Trend", q3_monthly)

# Query 4: Inventory vs Sales
q4_inventory = """
SELECT 
    p.ProductName,
    p.Category,
    p.StockQuantity,
    COALESCE(SUM(s.Quantity), 0) AS units_sold
FROM ProductItems p
LEFT JOIN SalesItems s ON p.ProductID = s.ProductID
GROUP BY p.ProductID
ORDER BY units_sold DESC
LIMIT 10;
"""
df_inventory = run_query("4. Top 10 Products Sold vs Inventory", q4_inventory)

# ==========================================
# 3. VISUALIZATIONS
# ==========================================

# Chart 1: Revenue by Category
plt.figure(figsize=(8, 4))
sns.barplot(data=df_category, x="revenue", y="Category", palette="Blues_r")
plt.title("Total Revenue by Product Category", fontsize=14, fontweight="bold")
plt.xlabel("Revenue ($)")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

# Chart 2: Monthly Sales Trend
if not df_monthly.empty and df_monthly["sales_month"].notna().any():
    plt.figure(figsize=(10, 4))
    sns.lineplot(
        data=df_monthly,
        x="sales_month",
        y="monthly_revenue",
        marker="o",
        color="#1f77b4",
        linewidth=2.5,
    )
    plt.title("Monthly Revenue Growth", fontsize=14, fontweight="bold")
    plt.xlabel("Month")
    plt.ylabel("Revenue ($)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()